# Рекомендация тарифов

В вашем распоряжении данные о поведении клиентов мобильного оператора. Нужно построить модель для задачи классификации, которая выберет подходящий тариф.

Постройте модель с максимально большим значением *accuracy* - нужно довести долю правильных ответов по крайней мере до 0.75. 
Проверьте *accuracy* на тестовой выборке самостоятельно.

**План работы:**
    
Шаг 1. Откроем файл с данными и изучим общую информацию
    
Шаг 2. Разделим исходные данные на обучающую, валидационную и тестовую выборки.
    
Шаг 3. Исследуем качество разных моделей, меняя гиперпараметры. Сформулируем выводы исследования.
    
Шаг 4. Проверим качество модели на тестовой выборке.

## Откройте и изучите файл

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from joblib import dump
import warnings
warnings.filterwarnings('ignore')

In [2]:
# напишите в этой ячейке код для чтения таблицы 'df' из файла 'users_behavior.csv'
df = pd.read_csv('users_behavior.csv')

In [3]:
# напишите в этой ячейке код для отображения информации о таблице
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [4]:
# напишите в этой ячейке код для вывода первых 10 строк таблицы 'df'
df.head(5)

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


Каждый объект в наборе данных — это информация о поведении одного пользователя за месяц. Известно:
* сalls — количество звонков,
* minutes — суммарная длительность звонков в минутах,
* messages — количество sms-сообщений,
* mb_used — израсходованный интернет-трафик в Мб,
* is_ultra — каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).

In [5]:
df['is_ultra'].value_counts()

is_ultra
0    2229
1     985
Name: count, dtype: int64

Определим признаки `features` и целевой признак `target `

In [6]:
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

**Вывод**

Каждая строка данных содержит информацию о поведении одного пользователя в течение месяца: 
* количество звонков, 
* суммарная длительность звонков в минутах, 
* количество sms-сообщений, 
* израсходованный интернет-трафик в Мб, 
* каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).

Для построения модели определили признаки: 
* количество звонков, 
* суммарная длительность звонков в минутах, 
* количество sms-сообщений, 
* израсходованный интернет-трафик в Мб.

Выбрали целевой признак - каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).

Количество абонентов тарифа "Ультра" в выборке меньше количества абонентов тарифа "Смарт" более, чем в 2 раза.

## Разбейте данные на выборки

Спрятанной тестовой выборки нет. Значит, данные нужно разбить на
три части: 
* обучающую, 
* валидационную и 
* тестовую. 

Размеры тестового и валидационного наборов обычно равны.

In [7]:
features_train, features_valid, target_train, target_valid = train_test_split(
   features, target, test_size=0.4, random_state=12345,stratify = target) 

In [8]:
features_valid, features_test, target_valid, target_test = train_test_split(features_valid, target_valid,  test_size=0.5, random_state=12345, stratify = target_valid)

**Вывод**

Исходные данные разбили в соотношении 3:1:1.

## Исследуйте модели

### Логистическая регрессия

In [9]:
best_model_logistic_reg = None
best_result_valid = 0
best_result_train = 0
solver_method=['lbfgs', 'newton-cg', 'liblinear', 'sag', 'saga']   # алгоритмы, используемые в логистической регрессии
for s in solver_method:
    model = LogisticRegression(random_state=12345, solver=s)       # меняем алгоритм оптимизации
    model_logistic_reg =model.fit(features_train, target_train)                             
    result_valid=model.score(features_valid, target_valid)
    result_train=model.score(features_train, target_train)
    print(f'Алгоритм = {s}')
    print(f'Accuracy на обучающей выборке: {result_train}')
    print(f'Accuracy на валидационной выборке: {result_valid}')
    if result_train > best_result_train:
        best_solver_train=s     # высота дерева для наилучшей модели
        best_model_logistic_reg_train = model   # наилучшая модель
        best_result_train = result_train # наилучшее значение метрики accuracy на обучающих данных
    if result_valid > best_result_valid:
        best_solver_valid=s     # высота дерева для наилучшей модели
        best_model_logistic_reg_valid = model   # наилучшая модель
        best_result_valid = result_valid # наилучшее значение метрики accuracy на валидационных данных
print('---------------------------------------------------------------------------')
print(f'Accuracy наилучшей модели логистической регрессии на обучающей выборке: {best_result_train}')
print(f'Алгоритм для лучшей модели на обучающей выборке = {best_solver_train}')
print(f'Accuracy наилучшей модели логистической регрессии на валидационной выборке: {best_result_valid}')
print(f'Алгоритм для лучшей модели на валидационной выборке = {best_solver_valid}')

Алгоритм = lbfgs
Accuracy на обучающей выборке: 0.7510373443983402
Accuracy на валидационной выборке: 0.7387247278382582
Алгоритм = newton-cg
Accuracy на обучающей выборке: 0.7510373443983402
Accuracy на валидационной выборке: 0.7387247278382582
Алгоритм = liblinear
Accuracy на обучающей выборке: 0.7105809128630706
Accuracy на валидационной выборке: 0.71850699844479
Алгоритм = sag
Accuracy на обучающей выборке: 0.6934647302904564
Accuracy на валидационной выборке: 0.6936236391912908
Алгоритм = saga
Accuracy на обучающей выборке: 0.6934647302904564
Accuracy на валидационной выборке: 0.6936236391912908
---------------------------------------------------------------------------
Accuracy наилучшей модели логистической регрессии на обучающей выборке: 0.7510373443983402
Алгоритм для лучшей модели на обучающей выборке = lbfgs
Accuracy наилучшей модели логистической регрессии на валидационной выборке: 0.7387247278382582
Алгоритм для лучшей модели на валидационной выборке = lbfgs


**Вывод**

На валидационной выборке лучшие результаты модели:
* алгоритм lbfgs имеет accuracy модели 0.7387247278382582.

## Проверьте модель на тестовой выборке

### Логистическая регрессия

В качестве модели для проверки на тестовой выборке возьмем модель логистической регрессии, которая показала лучший результат на валидационной выборке.

Так как выборка у нас небольшая, попробуем улучшить модель на расширенной выборке (обучающая+валидационная):

In [19]:
# объединим обучающую и валидационную выборки
features_train_new = features_train._append(features_valid, ignore_index=True)
target_train_new = target_train._append(target_valid, ignore_index=True)

AttributeError: 'DataFrame' object has no attribute '_append'

In [20]:
# обучим модель на расширенной выборке
model = LogisticRegression(random_state=12345, solver='lbfgs')       # записываем алгоритм оптимизации
model.fit(features_train_new, target_train_new)                             
result=model.score(features_train_new, target_train_new)
print(f'Accuracy на расширенной выборке: {result}')

NameError: name 'features_train_new' is not defined

Проверим модель на тестовой выборке:

In [16]:
answer = model.predict(features_test)

NotFittedError: This LogisticRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [17]:
result_logreg=accuracy_score(target_test,answer)
result_logreg

NameError: name 'answer' is not defined

In [18]:
print('Предсказания:',answer)
print('Правильные ответы:',target_test.values)

NameError: name 'answer' is not defined

**Вывод**

На тестовой выборке модель логистической регрессии показала  точность _________________